<a href="https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install datasets duckdb pandas pyarrow huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
from datasets import load_dataset

dim_clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients"
)

dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content"
)

fact_daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

fact_query = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d"
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [ ]:
fact_daily["train"].column_names

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row =** one content item's daily performance record for a specific date.

**Time window:** I will use a mid-panel month (for example, **2026-03**) to avoid using the final month as training data. This follows the assignment guidance and helps prevent information leakage from future outcomes.

In [ ]:
import pandas as pd

sample = pd.DataFrame(fact_daily["train"][:100000])

# Convert 'report_date' to datetime and extract the month in 'YYYY-MM' format
sample["month"] = pd.to_datetime(sample["report_date"]).dt.strftime('%Y-%m')

march = sample[sample["month"] == "2026-03"]

march.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- Clicks
- Impressions
- Average Position
- Device
- Country

These are available before making a prediction and describe historical search performance.

### Label / Proxy
Future organic clicks (or traffic growth) will be used as the prediction target.

### Context
- Date
- Month
- Content ID
- Client ID

These identify the observation but are not direct predictive features.

### Excluded
I exclude any future performance fields because they would leak information into the model and produce unrealistic evaluation results.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following queries verify:

1. The grain of the data.
2. The number of records in the selected month.
3. Data availability after filtering valid rows.

In [ ]:
march.head(10)

print("Rows:", len(march))

print("Date Range:")
print(march["report_date"].min())
print(march["report_date"].max())

available = march[march["gsc_data_available"] == True]

print(len(available))

Rows: 0
Date Range:
nan
nan
0


## Five Features

### 1. Clicks
Knowable at the decision moment because historical clicks already exist before prediction.

### 2. Impressions
Available before prediction since they are historical search metrics.

### 3. Average Position
Known from previous search performance.

### 4. Device
Available when the content is analysed.

### 5. Country
Known from historical traffic segmentation.

In [ ]:
feature_frame = march[
    [
        "gsc_clicks",
        "gsc_impressions",
        "gsc_avg_position",
        # "device", # Not found in fact_daily dataset
        # "country", # Not found in fact_daily dataset
    ]
]

feature_frame.head()

,gsc_clicks,gsc_impressions,gsc_avg_position


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset measures search performance rather than user intent or business outcomes.

It cannot explain why rankings changed or whether users converted after visiting the content.

Earlier records may contain limited Search Console coverage, and historical windows may overlap. Therefore, results should be treated as decision-support rather than proof of causation.

In [ ]:
# Honest feature set
features = feature_frame.copy()

# Deliberate leakage example (this column does not exist in the dataset)
# features["leak_feature"] = march["future_clicks"]

features.head()

,gsc_clicks,gsc_impressions,gsc_avg_position


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.